# 谷子 1844 种质 · 全体基因单倍型频率计算

对标 `coo-1-0-1-3-Rice_18k-get-hap-ratio.ipynb`（VCF → to_n_alt）与
`coo-1-0-2-2-Cowpea_344-get-cowpea-hap-ratio.ipynb`（文本直读）的统一逻辑，
适配 SetariDB 下载的 xls（实为 TSV）文件。

**算法**：基因区间内全部 SNP 按 POS 升序 → 样本×SNP 基因型矩阵 →
每样本跨 SNP 的基因型 tuple = 单倍型 → unique 按首次出现顺序编号 Hap1、Hap2、… →
count = 该单倍型样本数，percentage = count / 1844。

**已确认的设计决策**：
- 编码：值==REF→0、值==ALT→2、H→1、N→-1（与 Rice_18k 的 `to_n_alt()` 语义一致）
- 缺失 N：保留为独立状态 -1（与两个参考脚本语义一致）
- 输出文件名：gene_list.txt 的 File 列去掉 .xls，与 xls/ 一一对应

**输出**：
- `2-allGeneHapDf/{name}.txt`：样本×SNP 矩阵（行=sample，列=POS）+ haplotype_id，Tab，带行索引
- `2-allGeneHapPer/{name}.txt`：sample / haplotype_id / count / percentage，Tab，无行索引
- `hap_summary.csv`：每基因 SNP 数、单倍型数、最大单倍型占比汇总

**运行前提**（probe_result.csv 已验证）：
基因型取值仅 A/C/G/T/H/N；零多等位；零非法值；306 文件样本列完全一致（1844 个，
唯一、不含下划线）；POS 唯一且在区间内；TYPE 全为 SNP。

In [1]:
import pathlib

import numpy as np
import pandas as pd

# 路径全部相对当前工作目录，WSL / Windows Jupyter 均可直接运行
ROOT = pathlib.Path.cwd()
XLS_DIR = ROOT / "xls"
GENE_LIST_FILE = ROOT / "gene_list.txt"
HAPDF_PATH = ROOT / "2-allGeneHapDf"
HAPPER_PATH = ROOT / "2-allGeneHapPer"

HAPDF_PATH.mkdir(exist_ok=True)
HAPPER_PATH.mkdir(exist_ok=True)

N_META_COLS = 8  # CHROM POS REF ALT TYPE MAF MIS ANN

In [2]:
def build_hap_for_gene(row, base_samples):
    """对一个基因构建单倍型并输出两个文件，返回统计信息。

    row: dict，来自 gene_list.txt（GeneID/Symbol/Chromosome/Start/End/File）
    base_samples: 首文件确立的 1844 个样本列名，用于一致性断言
    """
    output_id = row["File"][:-4]  # 去掉 .xls
    src = XLS_DIR / row["File"]

    df = pd.read_csv(src, sep="\t", dtype=str, low_memory=False)

    # ---- 安全断言（任一触发即中止）----
    assert not df["ALT"].fillna("").str.contains(",").any(), f"{row['File']}: 存在多等位位点"
    assert set(df["TYPE"]) == {"SNP"}, f"{row['File']}: TYPE 不全为 SNP"

    pos = df["POS"].astype(int)
    assert not pos.duplicated().any(), f"{row['File']}: POS 存在重复"
    assert (pos >= int(row["Start"])).all() and (pos <= int(row["End"])).all(), \
        f"{row['File']}: POS 越出 gene_list 区间"
    assert set(df["CHROM"].astype(int)) == {int(row["Chromosome"])}, \
        f"{row['File']}: CHROM 与 gene_list 不一致"

    sample_cols = list(df.columns[N_META_COLS:])
    assert sample_cols == base_samples, f"{row['File']}: 样本列与首文件不一致"

    # ---- 按 POS 升序排序，保证单倍型组合顺序稳定 ----
    df = df.assign(_pos=pos).sort_values("_pos").drop(columns="_pos")
    pos_list = df["POS"].astype(int).tolist()

    ref = df["REF"].to_numpy()[:, None]
    alt = df["ALT"].to_numpy()[:, None]
    M = df[sample_cols].to_numpy()

    # ---- 向量化转换：值==REF→0、值==ALT→2、H→1、N→-1 ----
    G = np.full(M.shape, -99, dtype=np.int8)
    G[M == ref] = 0
    G[M == alt] = 2
    G[M == "H"] = 1
    G[M == "N"] = -1
    assert (G != -99).all(), f"{row['File']}: 存在既非 REF/ALT/H/N 的取值"

    # ---- 样本 × SNP 矩阵（1844 × n_snp）----
    GT = G.T
    geno_df = pd.DataFrame(GT, columns=pos_list, index=sample_cols)
    geno_df.index.name = "sample"

    # ---- 单倍型：每行的 SNP 组合 tuple ----
    hap_strings = [tuple(r) for r in GT.tolist()]
    hap_df = pd.DataFrame({"sample": sample_cols, "haplotype": hap_strings})

    # unique 按首次出现顺序编号 Hap1、Hap2、…
    unique_haps = hap_df["haplotype"].unique()
    hap_to_id = {h: f"Hap{idx + 1}" for idx, h in enumerate(unique_haps)}
    hap_df["haplotype_id"] = hap_df["haplotype"].map(hap_to_id)

    # count / percentage
    freq = hap_df.groupby("haplotype_id").size().reset_index(name="count")
    freq["percentage"] = freq["count"] / freq["count"].sum()
    freq2count = freq.set_index("haplotype_id")["count"].to_dict()
    freq2per = freq.set_index("haplotype_id")["percentage"].to_dict()
    hap_df["count"] = hap_df["haplotype_id"].map(freq2count)
    hap_df["percentage"] = hap_df["haplotype_id"].map(freq2per)

    sample2hap = hap_df[["sample", "haplotype_id"]].set_index("sample")["haplotype_id"].to_dict()
    geno_df["haplotype_id"] = geno_df.index.map(sample2hap)

    # ---- 输出 ----
    geno_df.to_csv(HAPDF_PATH / f"{output_id}.txt", sep="\t")
    hap_df[["sample", "haplotype_id", "count", "percentage"]].to_csv(
        HAPPER_PATH / f"{output_id}.txt", sep="\t", index=False)

    return {
        "File": row["File"], "GeneID": row["GeneID"], "Symbol": row["Symbol"],
        "n_snp": len(pos_list), "n_hap": len(unique_haps),
        "top1_count": int(freq["count"].max()),
        "top1_pct": float(freq["count"].max()) / len(sample_cols),
    }

In [3]:
info = pd.read_csv(GENE_LIST_FILE, sep="\t", dtype=str)
print(f"gene_list.txt: {len(info)} 行, 列: {info.columns.tolist()}")

# 首文件确立基准样本列
df0 = pd.read_csv(XLS_DIR / info.iloc[0]["File"], sep="\t", dtype=str,
                  low_memory=False, nrows=1)
base_samples = list(df0.columns[N_META_COLS:])
print(f"基准样本列: {len(base_samples)} 个, 示例 {base_samples[:3]} ... {base_samples[-2:]}")

results = []
t0 = pd.Timestamp.now()
for i, row in enumerate(info.to_dict("records"), 1):
    results.append(build_hap_for_gene(row, base_samples))
    if i % 30 == 0 or i == len(info):
        r = results[-1]
        print(f"  [{i}/{len(info)}] {r['Symbol']}: SNPs={r['n_snp']}, Haps={r['n_hap']}")

elapsed = pd.Timestamp.now() - t0
print(f"\nALL DONE: {len(results)} genes in {elapsed}")

gene_list.txt: 306 行, 列: ['GeneID', 'Symbol', 'Chromosome', 'Start', 'End', 'File']
基准样本列: 1844 个, 示例 ['WGS0083', 'WGS0320', 'WGS0323'] ... ['Yugu1', 'WGS0953']


  [30/306] ADF1: SNPs=20, Haps=108


  [60/306] FCA: SNPs=130, Haps=1338


  [90/306] NOM1: SNPs=18, Haps=108


  [120/306] DLT: SNPs=23, Haps=135


...

  [210/306] FT: SNPs=7, Haps=47


  [240/306] HD16: SNPs=51, Haps=266


  [270/306] PEP: SNPs=11, Haps=102


  [300/306] SWC6: SNPs=25, Haps=153


  [306/306] CRCK2: SNPs=35, Haps=220

ALL DONE: 306 genes in 0 days 00:00:24.270119


In [4]:
summary = pd.DataFrame(results)
summary.to_csv(ROOT / "hap_summary.csv", index=False, encoding="utf-8-sig")

print("=== 总体统计 ===")
print(f"基因数: {len(summary)}, 总 SNP: {int(summary.n_snp.sum()):,}")
print(f"单倍型数: 中位 {summary.n_hap.median():.0f}, 最大 {summary.n_hap.max()}, 最小 {summary.n_hap.min()}")
print(f"最大单倍型占比: 中位 {summary.top1_pct.median()*100:.1f}%")
print()
print("=== 按 SNP 数分档 ===")
bins = [0, 1, 5, 10, 20, 50, 100, 1000]
summary["bin"] = pd.cut(summary.n_snp, bins)
print(summary.groupby("bin", observed=True).agg(
    基因数=("n_hap", "size"),
    中位单倍型数=("n_hap", "median"),
    中位最大单倍型占比=("top1_pct", lambda s: f"{s.median()*100:.1f}%"),
).to_string())
print()
print("=== 单倍型数最多 / 最少的 5 个基因 ===")
print(summary.nlargest(5, "n_hap")[["Symbol", "n_snp", "n_hap", "top1_pct"]].round(3).to_string(index=False))
print()
print(summary.nsmallest(5, "n_hap")[["Symbol", "n_snp", "n_hap", "top1_pct"]].round(3).to_string(index=False))

=== 总体统计 ===
基因数: 306, 总 SNP: 17,461
单倍型数: 中位 232, 最大 1750, 最小 4
最大单倍型占比: 中位 38.6%

=== 按 SNP 数分档 ===
             基因数  中位单倍型数 中位最大单倍型占比
bin                               
(0, 1]         2     4.0     81.3%
(1, 5]        13    27.0     66.5%
(5, 10]       27    56.0     50.3%
(10, 20]      48   101.5     62.6%
(20, 50]      98   202.0     42.2%
(50, 100]     67   416.0     31.7%
(100, 1000]   51   804.0     18.7%

=== 单倍型数最多 / 最少的 5 个基因 ===
Symbol  n_snp  n_hap  top1_pct
  RFT1    213   1750     0.031
HMGB15    281   1673     0.051
   GW2    124   1652     0.026
   D27    125   1606     0.034
  ATRX    160   1458     0.072

Symbol  n_snp  n_hap  top1_pct
 SETH6      1      4     0.789
 EC1.1      1      4     0.837
  SE14      3     16     0.386
  LAX1      3     18     0.464
  EID1      3     19     0.784


In [5]:
from collections import Counter

expected = sorted(x[:-4] + ".txt" for x in info["File"])
hapdf_files = sorted(p.name for p in HAPDF_PATH.glob("*.txt"))
happer_files = sorted(p.name for p in HAPPER_PATH.glob("*.txt"))

checks = [
    ("HapDf 文件数 = 306", len(hapdf_files) == len(expected) == 306),
    ("HapPer 文件数 = 306", len(happer_files) == len(expected) == 306),
    ("HapDf 文件名与 gene_list 一一对应", hapdf_files == expected),
    ("HapPer 文件名与 gene_list 一一对应", happer_files == expected),
]

# 全量检查（注意语义：HapPer 每样本一行，count 为该样本所属单倍型的样本总数，
# 直接求和会得到 sum(count^2)，必须按 haplotype_id 去重后再求和）
bad = []
for fn in happer_files:
    per = pd.read_csv(HAPPER_PATH / fn, sep="\t")
    if len(per) != 1844:
        bad.append((fn, "rows", len(per)))
    if per.groupby("haplotype_id")["count"].nunique().gt(1).any():
        bad.append((fn, "count_inconsistent"))
    u = per.drop_duplicates("haplotype_id")
    if u["count"].sum() != 1844:
        bad.append((fn, "count_sum", int(u["count"].sum())))
    if abs(u["percentage"].sum() - 1.0) > 1e-9:
        bad.append((fn, "per_sum", u["percentage"].sum()))
checks.append(("全部 HapPer: 1844 行 / 同hap内count一致 / 去重count和=1844 / percentage和=1", not bad))
if bad:
    print("  异常示例:", bad[:5])

# 抽查 3 个基因（n_snp 最小 / 最接近中位 / 最大）：独立重算 + HapDf 交叉验证
def independent_recount(fname):
    """独立于 build_hap_for_gene 的重算路径：直接从源 xls 重新编码并计数"""
    df = pd.read_csv(XLS_DIR / fname, sep="\t", dtype=str, low_memory=False)
    sc = list(df.columns[N_META_COLS:])
    order = np.argsort(df["POS"].astype(int).to_numpy())
    df = df.iloc[order]
    ref = df["REF"].to_numpy()
    alt = df["ALT"].to_numpy()
    M = df[sc].to_numpy()
    G = np.full(M.shape, -99, dtype=np.int8)
    G[M == ref[:, None]] = 0
    G[M == alt[:, None]] = 2
    G[M == "H"] = 1
    G[M == "N"] = -1
    assert (G != -99).all()
    return Counter(tuple(r) for r in G.T.tolist())

mid_i = (summary.n_snp - summary.n_snp.median()).abs().argsort().iloc[0]
picks = [summary.nsmallest(1, "n_snp").iloc[0],
         summary.iloc[mid_i],
         summary.nlargest(1, "n_snp").iloc[0]]
for p in picks:
    cnt = independent_recount(p["File"])
    per = pd.read_csv(HAPPER_PATH / (p["File"][:-4] + ".txt"), sep="\t")
    u = per.drop_duplicates("haplotype_id")
    ok1 = sorted(cnt.values()) == sorted(u["count"].tolist())   # 单倍型计数分布一致
    ok2 = len(cnt) == len(u)                                    # 单倍型个数一致
    dfh = pd.read_csv(HAPDF_PATH / (p["File"][:-4] + ".txt"), sep="\t", index_col=0)
    ok3 = dfh.shape == (1844, int(p["n_snp"]) + 1)             # 矩阵形状
    # 最强验证：从 HapDf 矩阵重建单倍型计数 == 独立重算
    cnt_hapdf = Counter(tuple(r) for r in dfh.drop(columns="haplotype_id").to_numpy().tolist())
    ok4 = cnt_hapdf == cnt
    # HapDf 与 HapPer 的逐样本 haplotype_id 完全一致
    j = per.set_index("sample")["haplotype_id"]
    ok5 = (dfh["haplotype_id"] == j.loc[dfh.index]).all()
    checks.append((f"抽查 {p['Symbol']} (SNP={int(p['n_snp'])}): 独立重算/矩阵重建/双文件一致",
                   bool(ok1 and ok2 and ok3 and ok4 and ok5)))

print("=== 验收 ===")
allpass = True
for desc, okv in checks:
    allpass &= bool(okv)
    print(f"  [{'PASS' if okv else 'FAIL'}] {desc}")
print()
print("验收结论:", "全部通过" if allpass else "存在失败项")

=== 验收 ===
  [PASS] HapDf 文件数 = 306
  [PASS] HapPer 文件数 = 306
  [PASS] HapDf 文件名与 gene_list 一一对应
  [PASS] HapPer 文件名与 gene_list 一一对应
  [PASS] 全部 HapPer: 1844 行 / 同hap内count一致 / 去重count和=1844 / percentage和=1
  [PASS] 抽查 SETH6 (SNP=1): 独立重算/矩阵重建/双文件一致
  [PASS] 抽查 OsI_33462 (SNP=38): 独立重算/矩阵重建/双文件一致
  [PASS] 抽查 KIN1 (SNP=379): 独立重算/矩阵重建/双文件一致

验收结论: 全部通过
